In [1]:
# 01_data_loading.ipynb
# Regulatory-Constrained Fairness Score for Credit Scoring
# Notebook 01 - Data loading and target construction.
#
# Loads three public credit datasets and fixes, explicitly, the sensitive
# attributes, the privileged reference groups, and the favourable outcome used
# throughout the study:
#   German Credit (UCI/Statlog) - target = 1 if "bad" (default); favourable
#       outcome = NON-default (loan would perform). Sensitive: sex, age_group.
#       Privileged reference: sex = male, age_group = old (age >= 25).
#   Taiwan Credit Card Default (UCI) - target = 1 if default next month;
#       favourable outcome = NON-default. Sensitive: sex, education.
#       Privileged reference: sex = male, education in {graduate, university}.
#   Credit Card Approval (Kaggle) - target built from the monthly STATUS record
#       at a 30-days-past-due threshold; favourable outcome = NON-default.
#       Sensitive: sex. Privileged reference: sex = male.
# All sensitive attributes are kept as plain columns so that fairness tooling
# downstream can consume them. No filesystem paths are printed (anonymity).

import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Paths are relative to the notebooks/ folder and are never printed.
DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

# The favourable (desirable) decision for an applicant is a NON-default
# prediction, i.e. the loan is expected to perform. Because every dataset codes
# target = 1 for the adverse event (default), favourable corresponds to y = 0.
FAVOURABLE_LABEL = 0  # documented once, consumed by later notebooks via meta

# --- German Credit (Statlog) ---
# Whitespace-separated, no header, coded categorical values.
german_cols = [
    "status_checking", "duration", "credit_history", "purpose", "credit_amount",
    "savings", "employment", "installment_rate", "personal_status_sex", "other_debtors",
    "residence_since", "property", "age", "other_installment", "housing",
    "existing_credits", "job", "num_dependents", "telephone", "foreign_worker", "target",
]
df_german = pd.read_csv(
    os.path.join(DATA_DIR, "german.data"), sep=r"\s+", header=None, names=german_cols
)
# Original target: 1 = good, 2 = bad -> recode 1 = bad (default), 0 = good.
df_german["target"] = (df_german["target"] == 2).astype(int)
# Sensitive attributes. personal_status_sex encodes sex jointly with status.
sex_map = {"A91": "male", "A92": "female", "A93": "male", "A94": "male", "A95": "female"}
df_german["sex"] = df_german["personal_status_sex"].map(sex_map)
# Age dichotomised at 25 (common Statlog fairness convention); old = privileged.
df_german["age_group"] = np.where(df_german["age"] >= 25, "old", "young")
print("German Credit:", df_german.shape, "| default rate:", round(df_german["target"].mean(), 3))

# --- Taiwan Credit Card Default ---
# .xls with a title row -> header=1 (requires xlrd >= 2.0.1).
df_taiwan = pd.read_excel(
    os.path.join(DATA_DIR, "default of credit card clients.xls"), header=1
)
df_taiwan = df_taiwan.rename(columns={"default payment next month": "target"})
if "ID" in df_taiwan.columns:
    df_taiwan = df_taiwan.drop(columns=["ID"])
df_taiwan["sex"] = df_taiwan["SEX"].map({1: "male", 2: "female"})
df_taiwan["education"] = df_taiwan["EDUCATION"].map({
    1: "graduate", 2: "university", 3: "high_school",
    0: "others", 4: "others", 5: "others", 6: "others",
})
print("Taiwan CC Default:", df_taiwan.shape, "| default rate:", round(df_taiwan["target"].mean(), 3))

# --- Credit Card Approval (Kaggle) ---
# application_record.csv (features) + credit_record.csv (monthly STATUS).
# STATUS: 0=1-29d, 1=30-59d, 2=60-89d, 3=90-119d, 4=120-149d, 5=150+d/writeoff,
#         C=paid off, X=no loan. Label at 30+ dpd: bad if any month STATUS>=1.
PRIMARY_TARGET = "target_30dpd"
df_app = pd.read_csv(os.path.join(DATA_DIR, "application_record.csv"))
df_credit = pd.read_csv(os.path.join(DATA_DIR, "credit_record.csv"))
status_num = df_credit["STATUS"].replace({"C": -1, "X": -1}).astype(int)
df_credit = df_credit.assign(status_num=status_num)
worst = df_credit.groupby("ID")["status_num"].max()
labels = pd.DataFrame({
    "target_30dpd": (worst >= 1).astype(int),
    "target_60dpd": (worst >= 2).astype(int),
})
df_home = df_app.merge(labels, on="ID", how="inner")
df_home["target"] = df_home[PRIMARY_TARGET]
df_home["sex"] = df_home["CODE_GENDER"].map({"M": "male", "F": "female"})
df_home = df_home[df_home["sex"].notna()].copy()
print("Credit Card Approval:", df_home.shape, "| default rate:", round(df_home["target"].mean(), 3))

# --- Dataset summary table (Table 1) ---
summary = pd.DataFrame([
    {"dataset": "German Credit", "n_rows": len(df_german), "n_cols": df_german.shape[1],
     "pos_rate": round(df_german["target"].mean(), 3),
     "sensitive": "sex, age_group", "favourable": "non-default (y=0)"},
    {"dataset": "Taiwan CC Default", "n_rows": len(df_taiwan), "n_cols": df_taiwan.shape[1],
     "pos_rate": round(df_taiwan["target"].mean(), 3),
     "sensitive": "sex, education", "favourable": "non-default (y=0)"},
    {"dataset": "Credit Card Approval", "n_rows": len(df_home), "n_cols": df_home.shape[1],
     "pos_rate": round(df_home["target"].mean(), 3),
     "sensitive": "sex", "favourable": "non-default (y=0)"},
])
print(summary.to_string(index=False))
summary.to_csv(os.path.join(TAB_DIR, "01_dataset_summary.csv"), index=False)

# --- Persist raw frames for the preprocessing notebook ---
df_german.to_parquet(os.path.join(PROCESSED_DIR, "german.parquet"))
df_taiwan.to_parquet(os.path.join(PROCESSED_DIR, "taiwan.parquet"))
df_home.to_parquet(os.path.join(PROCESSED_DIR, "credit_card_approval.parquet"))
print("Notebook 01 complete.")


German Credit: (1000, 23) | default rate: 0.3


Taiwan CC Default: (30000, 26) | default rate: 0.221


Credit Card Approval: (36457, 22) | default rate: 0.118
             dataset  n_rows  n_cols  pos_rate      sensitive        favourable
       German Credit    1000      23     0.300 sex, age_group non-default (y=0)
   Taiwan CC Default   30000      26     0.221 sex, education non-default (y=0)
Credit Card Approval   36457      22     0.118            sex non-default (y=0)
Notebook 01 complete.
